# Building a new `cafpybara` analysis, file by file

Walks through `analyses/_template/` bare-minimum first, then each
customization layered on.

## 0. Make a new analysis directory

Copy `analyses/_template/` as `analyses/my_analysis/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../../../..")))

import pandas as pd
from cafpybara import core

# TODO: import your own analysis's name instead of _template.
import cafpybara.analyses._template as ana

STILL_ON_TEMPLATE = ana.__name__.endswith('_template')
if STILL_ON_TEMPLATE:
    print("Still importing _template directly -- see the TODO above.")
else:
    print("Imported cleanly -- a real, importable analysis.")

In [ ]:
# Define once, used throughout this notebook:
# MC_FILE must be produced by cafpyana first, e.g. via
# `run_df_maker.py -c configs/hnl_mcnu_nopreselect_savepfp.py -l <input.list> -o <output_name>`
# (no weights, no preselection, all PFPs saved -- see cafpyana's hnl_nuee_nupi0 README.
MC_FILE = "your_mc_file.h5"
VAR = "your_variable"
BINS = [0, 1, 2, 3]

## 1. `__init__.py`

* TODO: none -- keep `__init__.py` as-is.

Check below that everything resolves:

In [ ]:
modules_to_check = ['io', 'funcs', 'plotting', 'selection', 'physics', 'syst', 'classes', 'preprocess', 'detvar']
for mod_name in modules_to_check:
    core_mod = getattr(core, mod_name)
    missing = [n for n in getattr(core_mod, '__all__', []) if not hasattr(ana, n)]
    status = 'OK' if not missing else f'MISSING: {missing}'
    print(f"core.{mod_name:12s} -> {status}")

## 2. Bare minimum: load + plot MC

1. Load in MC files with `ana.load_mc(...)`:

* TODO: Set `REC_KEY` in `io.py` (currently `"TODO_SET_ME"`) e.g. `'rec'` or `'nuecc'`.
* Arguments: `file` -- a dataframe produced by cafpyana.
* Returns:
  * `df`: dataframe
  * `pot`: accumulated POT
  * `ngen`: number of generated events

2. Plot MC with `ana.plot_var`:
* Arguments: variable name + binning

See below for the actual call:

In [ ]:
# What you'd actually write in your own analysis notebook:
try:
    df, pot, ngen = ana.load_mc(MC_FILE)
    ana.plot_var(df, VAR, bins=BINS)
except Exception as e:
    print("No real MC file here (this notebook needs none) -- expected:")
    print(f"  {type(e).__name__}: {e}")

## 3. `analysis.py` -- categories

* TODO: set `signal_categories`/`signal_dict` -- used by `ana.plot_var`
  and the `signal` column `define_signal` computes.

See below for the actual call:

In [ ]:
print("signal_categories keys:", list(ana.signal_categories.keys()))
print("signal_dict:", ana.signal_dict)

In [ ]:
# What you'd actually write in your own analysis notebook:
try:
    df, pot, ngen = ana.load_mc(MC_FILE)
    ana.plot_var(df, VAR, bins=BINS, categories=ana.signal_categories)
except Exception as e:
    print("No real MC file here (this notebook needs none) -- expected:")
    print(f"  {type(e).__name__}: {e}")

## 4. `analysis.py` -- cuts

* TODO: set `DEFAULT_CUTS` -- pass to `load_mc(cuts=...)` or
  `select()` to apply it.

See below for the actual call:

In [ ]:
print("DEFAULT_CUTS:")
for c in ana.DEFAULT_CUTS:
    print(f"  {c.name!r}: {c.label}")

In [ ]:
# What you'd actually write in your own analysis notebook:
try:
    df, pot, ngen = ana.load_mc(MC_FILE, cuts=ana.DEFAULT_CUTS)
    ana.plot_var(df, VAR, bins=BINS, categories=ana.signal_categories)
except Exception as e:
    print("No real MC file here (this notebook needs none) -- expected:")
    print(f"  {type(e).__name__}: {e}")

## 5. `preprocess.py` -- adding preprocessing (Optional)

Preprocessing can cover fixing existing columns (e.g. timing
calibration) or adding new derived variables (e.g. shower angles).

### Fixing columns

* TODO: add real fixes in `preprocess_mc`.

Example -- HNL/pi0's MC timing calibration
(`analyses/hnlpi0/preprocess.py`):

```python
from ...core.preprocess import fix_timing_calibration
from ...core import timing_calibration as tc

def preprocess_mc(df):
    df = _core_preprocess_mc(df)
    df = fix_timing_calibration(df, period=tc.mcbnb_period_calib,
                                 t0_offset=tc.mcbnb_offset_calib)
    df = add_variables(df)
    return df
```

* Input: `slc.barycenterFM.flashTime` + `slc.vertex.z`, plus your
  sample's `period`/`t0_offset` calibration constants
  (`core.timing_calibration`).
* Output: adds `slc.barycenterFM.flashTime_calib` (+ period-folded
  `_calib_mod`) columns.

See below for the actual call:

In [ ]:
# What you'd actually write in your own analysis notebook:
try:
    df, pot, ngen = ana.load_mc(MC_FILE, preprocess_fn=ana.preprocess.preprocess_mc)
    ana.plot_var(df, VAR, bins=BINS)
except Exception as e:
    print("No real MC file here (this notebook needs none) -- expected:")
    print(f"  {type(e).__name__}: {e}")

### Adding new variables

* TODO: add `core.preprocess.add_variables(df)` to `preprocess_mc`/
  `preprocess_data` if your topology needs its derived shower-angle
  columns -- not called by default.

Example -- HNL/pi0 calls it in every bundler
(`analyses/hnlpi0/preprocess.py`):

```python
def preprocess_mc(df):
    df = _core_preprocess_mc(df)
    df = add_variables(df)
    return df
```

See source below:

In [ ]:
import inspect
print(inspect.getsource(core.preprocess.add_variables))